<a href="https://colab.research.google.com/github/erreesse-rs/trabalho-api/blob/main/projeto_marvel_dc_powerbi_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto Marvel x DC — Coleta e preparação para Power BI

Este notebook realiza:
1. Coleta de personagens da Marvel e DC.
2. Enriquecimento de uma amostra ou de todos os personagens com equipes, poderes e criadores.
3. Coleta de séries (`volumes`) das duas editoras.
4. Recuperação dos `concepts` associados às séries para análise temática.
5. Exportação de tabelas CSV relacionais para o Power BI.

**Importante sobre a temática:** no Comic Vine, os `concepts` são associados aos *volumes* (séries de quadrinhos), não diretamente ao personagem. Por isso, a dimensão temática fica em `volumes.csv`, `concepts.csv` e `volume_concepts.csv`.

O código usa cache JSON e checkpoints para reduzir requisições repetidas. Comece com os limites de teste da célula de configuração; depois aumente gradualmente.

## 1. Bibliotecas e autenticação

A API Key é solicitada de forma oculta. Não escreva a chave diretamente no notebook que será publicado.

In [ ]:
import os
import re
import json
import time
import html
import shutil
import requests
import pandas as pd

from pathlib import Path
from getpass import getpass
from IPython.display import display

#conectar o Google Drive ao Colab
from google.colab import drive
drive.mount('/content/drive')

# Definir a pasta onde todos os dados serão armazenados
PASTA_DADOS = Path(
    "/content/drive/MyDrive/dados_comicvine/dados_comicvine"
)
# Criar a pasta caso ela ainda não exista
PASTA_DADOS.mkdir(parents=True, exist_ok=True)

print(f"Arquivos serão salvos em: {PASTA_DADOS}")

# Solicitação de acesso a API
API_KEY = getpass("Digite sua API Key do Comic Vine: ")

if not API_KEY.strip():
    raise ValueError("A API Key não pode estar vazia.")


Mounted at /content/drive
Arquivos serão salvos em: /content/drive/MyDrive/dados_comicvine/dados_comicvine
Digite sua API Key do Comic Vine: ··········


## 2. Classe `ComicVine`

A classe centraliza as requisições, respeita o intervalo entre chamadas, trata erros temporários, mantém backups e permite retomar coletas.

Os CSVs brutos são gravados com `;` e UTF-8 sem BOM durante a gravação incremental. O BOM é aplicado somente nas exportações finais.

In [ ]:
class ComicVine:

    def __init__(
        self,
        api_key,
        intervalo=3,
        pasta="dados_comicvine"
    ):
        self.url = "https://comicvine.gamespot.com/api/"
        self.api_key = api_key
        self.intervalo = intervalo
        self.pasta = Path(pasta)
        self.pasta.mkdir(parents=True, exist_ok=True)

        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": "ProjetoCienciaDados/1.0"
        })

        self.ultima_requisicao = 0

    # --------------------------------------
    # Requisições HTTP e tratamento de erros
    # --------------------------------------

    def _requisitar(self, endpoint, parametros=None, max_tentativas=3):

        parametros = (parametros or {}).copy()
        parametros["api_key"] = self.api_key
        parametros["format"] = "json"

        url = self.url + endpoint

        for tentativa in range(1, max_tentativas + 1):

            tempo_decorrido = time.monotonic() - self.ultima_requisicao
            espera_intervalo = self.intervalo - tempo_decorrido

            if espera_intervalo > 0:
                time.sleep(espera_intervalo)

            try:
                resposta = self.session.get(
                    url,
                    params=parametros,
                    timeout=(10, 90)
                )
            except requests.exceptions.RequestException as erro:
                self.ultima_requisicao = time.monotonic()

                if tentativa < max_tentativas:
                    espera = 5 * tentativa
                    print(f"Falha de conexão. Nova tentativa em {espera}s.")
                    time.sleep(espera)
                    continue

                raise RuntimeError(
                    f"Erro de conexão após {max_tentativas} tentativas: {erro}"
                ) from erro

            self.ultima_requisicao = time.monotonic()

            # Erros transitórios do servidor
            if resposta.status_code in (502, 503, 504):
                if tentativa < max_tentativas:
                    espera = 5 * tentativa
                    print(
                        f"Servidor temporariamente indisponível "
                        f"(HTTP {resposta.status_code}). "
                        f"Nova tentativa em {espera}s."
                    )
                    time.sleep(espera)
                    continue

                raise RuntimeError(
                    f"Servidor indisponível após {max_tentativas} tentativas. "
                    f"HTTP {resposta.status_code}."
                )

            # Limite de requisições
            if resposta.status_code in (420, 429):
                raise RuntimeError(
                    "RATE LIMIT: a API bloqueou temporariamente as requisições. "
                    "Interrompa a coleta e aguarde antes de tentar novamente."
                )

            # Autenticação
            if resposta.status_code == 401:
                raise RuntimeError("API Key inválida ou não autorizada.")

            # Outros erros HTTP
            if not resposta.ok:
                raise RuntimeError(
                    f"Erro HTTP {resposta.status_code}: {resposta.text[:500]}"
                )

            # JSON
            try:
                dados = resposta.json()
            except ValueError as erro:
                raise RuntimeError(
                    f"A API retornou conteúdo que não é JSON. "
                    f"HTTP {resposta.status_code}."
                ) from erro

            status_api = dados.get("status_code")

            if status_api == 107:
                raise RuntimeError(
                    "RATE LIMIT informado pela API. Aguarde antes de continuar."
                )

            if status_api == 100:
                raise RuntimeError("A API informou que a chave é inválida.")

            if status_api != 1:
                raise RuntimeError(
                    f"Erro da API: {dados.get('error')} "
                    f"(código {status_api})."
                )

            return dados.get("results", [])

        raise RuntimeError("A requisição terminou sem resultado.")

    # --------------------------------------
    # Utilitários
    # --------------------------------------

    @staticmethod
    def _nome_recurso(valor):
        if isinstance(valor, dict):
            return valor.get("name")
        return valor

    @staticmethod
    def _id_recurso(valor):
        if isinstance(valor, dict):
            return valor.get("id")
        return None

    @staticmethod
    def _url_imagem(valor):
        if isinstance(valor, dict):
            return valor.get("medium_url") or valor.get("original_url")
        return valor

    def _fazer_backup(self, caminho):
        caminho = Path(caminho)

        if caminho.exists():
            destino = Path(str(caminho) + ".backup")
            contador = 1

            while destino.exists():
                destino = Path(f"{caminho}.backup_{contador}")
                contador += 1

            os.replace(caminho, destino)
            print(f"Backup criado: {destino}")

    @staticmethod
    def _detectar_separador(caminho):
        # Compatibilidade com CSVs antigos separados por vírgula
        # e CSVs atuais separados por ponto e vírgula.
        with open(caminho, "r", encoding="utf-8-sig", newline="") as arquivo:
            primeira_linha = arquivo.readline()

        return ";" if primeira_linha.count(";") > primeira_linha.count(",") else ","

    def _ler_csv_existente(self, caminho):
        separador = self._detectar_separador(caminho)

        df = pd.read_csv(
            caminho,
            sep=separador,
            encoding="utf-8-sig",
            dtype={"id": str, "publisher_id": str},
            low_memory=False
        )

        df.columns = [
            str(coluna).replace("\ufeff", "").strip()
            for coluna in df.columns
        ]

        for coluna in df.select_dtypes(include="object").columns:
            df[coluna] = df[coluna].str.replace("\ufeff", "", regex=False)

        return df

    @staticmethod
    def _salvar_checkpoint(caminho, dados):
        caminho = Path(caminho)
        temporario = Path(str(caminho) + ".tmp")

        with open(temporario, "w", encoding="utf-8") as arquivo:
            json.dump(dados, arquivo, ensure_ascii=False, indent=2)

        os.replace(temporario, caminho)

    # --------------------------------------
    # Teste de conexão e editoras
    # --------------------------------------

    def testar_conexao(self):
        resultado = self._requisitar(
            "characters/",
            {"limit": 1, "field_list": "id,name"}
        )
        df = pd.DataFrame(resultado)
        print("Conexão com Comic Vine realizada.")
        return df

    def buscar_editoras(self):
        resultado = self._requisitar(
            "publishers/",
            {"limit": 100, "field_list": "id,name"}
        )
        df = pd.DataFrame(resultado)

        if not df.empty:
            df = df[["id", "name"]]

        return df

    # --------------------------------------
    # Organização dos personagens
    # --------------------------------------

    def organizar_personagem(self, personagem):

        primeira_edicao = personagem.get("first_appeared_in_issue")

        if isinstance(primeira_edicao, dict):
            primeira_edicao = primeira_edicao.get("name")

        aliases = personagem.get("aliases")

        if isinstance(aliases, list):
            aliases = " | ".join(
                item.get("name", "") if isinstance(item, dict) else str(item)
                for item in aliases
            )

        publisher = personagem.get("publisher")
        origin = personagem.get("origin")
        image = personagem.get("image")

        return {
            "id": personagem.get("id"),
            "name": personagem.get("name"),
            "real_name": personagem.get("real_name"),
            "aliases": aliases,
            "gender": personagem.get("gender"),
            "birth": personagem.get("birth"),
            "publisher": self._nome_recurso(publisher),
            "publisher_id": self._id_recurso(publisher),
            "origin": self._nome_recurso(origin),
            "deck": personagem.get("deck"),
            "image_url": self._url_imagem(image),
            "count_of_issue_appearances": personagem.get(
                "count_of_issue_appearances"
            ),
            "first_appeared_in_issue": primeira_edicao,
            "date_added": personagem.get("date_added"),
            "date_last_updated": personagem.get("date_last_updated")
        }

    # --------------------------------------
    # Coleta de personagens por editora
    # --------------------------------------

    def coletar_personagens(
        self,
        id_editora,
        quantidade=500,
        nome_arquivo=None,
        retomar=True
    ):

        id_editora = int(id_editora)
        quantidade = int(quantidade)

        if quantidade <= 0:
            raise ValueError("quantidade deve ser maior que zero.")

        if nome_arquivo is None:
            nome_arquivo = f"personagens_{id_editora}.csv"

        caminho_csv = self.pasta / nome_arquivo
        caminho_checkpoint = Path(str(caminho_csv) + ".json")

        personagens = []
        ids_coletados = set()
        offset = 0

        tem_csv = caminho_csv.exists()
        tem_checkpoint = caminho_checkpoint.exists()

        # Retomada somente quando CSV e checkpoint existem juntos
        if retomar and tem_csv and tem_checkpoint:
            try:
                with open(caminho_checkpoint, "r", encoding="utf-8") as arquivo:
                    progresso = json.load(arquivo)

                if int(progresso.get("id_editora", -1)) != id_editora:
                    raise ValueError("Checkpoint pertence a outra editora.")

                df_existente = self._ler_csv_existente(caminho_csv)

                obrigatorias = {"id", "publisher_id"}
                if not obrigatorias.issubset(df_existente.columns):
                    raise ValueError("CSV não contém id e publisher_id.")

                if df_existente.empty:
                    raise ValueError("CSV existente está vazio.")

                # Migra arquivos antigos: a coluna de imagem era chamada `image`.
                if "image_url" not in df_existente.columns and "image" in df_existente.columns:
                    df_existente = df_existente.rename(
                        columns={"image": "image_url"}
                    )

                # Mantém exatamente o mesmo esquema nas linhas antigas e novas.
                colunas_personagem = [
                    "id", "name", "real_name", "aliases", "gender",
                    "birth", "publisher", "publisher_id", "origin",
                    "deck", "image_url", "count_of_issue_appearances",
                    "first_appeared_in_issue", "date_added",
                    "date_last_updated"
                ]

                for coluna in colunas_personagem:
                    if coluna not in df_existente.columns:
                        df_existente[coluna] = pd.NA

                df_existente = df_existente[colunas_personagem]

                ids_editoras = pd.to_numeric(
                    df_existente["publisher_id"],
                    errors="coerce"
                )

                if ids_editoras.isna().any():
                    raise ValueError("Há publisher_id ausente ou inválido.")

                if set(ids_editoras.astype(int)) != {id_editora}:
                    raise ValueError("CSV contém personagens de outra editora.")

                if df_existente["id"].isna().any():
                    raise ValueError("Há personagens sem ID.")

                df_existente = df_existente.drop_duplicates(
                    subset="id",
                    keep="first"
                )

                # Normaliza o arquivo antigo para o separador atual.
                df_existente["publisher_id"] = id_editora
                df_existente.to_csv(
                    caminho_csv,
                    index=False,
                    sep=";",
                    encoding="utf-8"
                )

                personagens = df_existente.to_dict(orient="records")
                ids_coletados = set(df_existente["id"].astype(str))
                offset = int(progresso.get("offset", -1))

                if offset < 0:
                    raise ValueError("Offset inválido no checkpoint.")

                print(
                    f"Retomando editora {id_editora}: "
                    f"{len(ids_coletados)} personagens já salvos."
                )

            except Exception as erro:
                print(f"Não foi possível retomar: {erro}")
                print("Os arquivos antigos serão preservados em backup.")
                self._fazer_backup(caminho_csv)
                self._fazer_backup(caminho_checkpoint)
                personagens = []
                ids_coletados = set()
                offset = 0

        else:
            # Coleta nova ou arquivos incompletos
            self._fazer_backup(caminho_csv)
            self._fazer_backup(caminho_checkpoint)

        if len(ids_coletados) >= quantidade:
            print("Quantidade solicitada já foi coletada.")
            return pd.DataFrame(personagens).head(quantidade)

        field_list = (
            "id,name,real_name,aliases,gender,birth,publisher,origin,"
            "deck,image,count_of_issue_appearances,"
            "first_appeared_in_issue,date_added,date_last_updated"
        )

        while len(ids_coletados) < quantidade:

            parametros = {
                "limit": 100,
                "offset": offset,
                "sort": "count_of_issue_appearances:desc",
                "field_list": field_list
            }

            print(f"Buscando personagens a partir do offset {offset}...")

            try:
                resultados = self._requisitar("characters/", parametros)
            except RuntimeError as erro:
                print(f"Coleta interrompida: {erro}")
                print("Os dados já salvos foram preservados.")
                break

            if not isinstance(resultados, list) or not resultados:
                print("Fim dos resultados ou resposta vazia.")
                break

            novos = []
            itens_processados = 0

            for indice, personagem in enumerate(resultados):
                itens_processados = indice + 1

                if not isinstance(personagem, dict):
                    continue

                publisher = personagem.get("publisher")
                personagem_id = personagem.get("id")

                if not isinstance(publisher, dict) or personagem_id is None:
                    continue

                try:
                    publisher_id = int(publisher.get("id"))
                except (TypeError, ValueError):
                    continue

                if publisher_id != id_editora:
                    continue

                personagem_id = str(personagem_id)

                if personagem_id in ids_coletados:
                    continue

                registro = self.organizar_personagem(personagem)
                registro["publisher_id"] = id_editora

                ids_coletados.add(personagem_id)
                novos.append(registro)

                if len(ids_coletados) >= quantidade:
                    break

            if novos:
                df_novos = pd.DataFrame(novos)
                arquivo_existe = caminho_csv.exists()

                df_novos.to_csv(
                    caminho_csv,
                    mode="a" if arquivo_existe else "w",
                    header=not arquivo_existe,
                    index=False,
                    sep=";",
                    encoding="utf-8"
                )

                personagens.extend(novos)

            offset += itens_processados

            self._salvar_checkpoint(
                caminho_checkpoint,
                {
                    "id_editora": id_editora,
                    "offset": offset,
                    "total_coletado": len(ids_coletados)
                }
            )

            print(
                f"Personagens válidos: "
                f"{len(ids_coletados)}/{quantidade}"
            )

            if len(ids_coletados) >= quantidade:
                break

            if len(resultados) < 100:
                print("Última página disponível.")
                break

        df_final = pd.DataFrame(personagens)

        if not df_final.empty:
            df_final = df_final.drop_duplicates(subset="id", keep="first")

        print(
            f"Coleta da editora {id_editora} finalizada/interrompida: "
            f"{len(df_final)} personagens."
        )

        return df_final.head(quantidade)

    # --------------------------------------
    # Detalhes dos personagens: equipes,
    # poderes e criadores (sem aliados/inimigos)
    # --------------------------------------


    def buscar_detalhes_personagem(self, id_personagem):

        campos = (
            "id,name,publisher,image,"
            "teams,powers,creators"
        )

        # 4005 é o prefixo correto para personagens
        resultado = self._requisitar(
            f"character/4005-{int(id_personagem)}/",
            {
                "field_list": campos
            }
        )

        if isinstance(resultado, list):
            return resultado[0] if resultado else None

        if isinstance(resultado, dict):
            return resultado

        return None

    def enriquecer_personagens(
        self,
        df,
        limite=None,
        pasta_cache="detalhes_personagens"
    ):

        if not isinstance(df, pd.DataFrame):
            raise TypeError("Informe um DataFrame.")

        if "id" not in df.columns:
            raise ValueError("A base não possui a coluna id.")

        pasta = self.pasta / pasta_cache
        pasta.mkdir(parents=True, exist_ok=True)

        ids = (
            pd.to_numeric(df["id"], errors="coerce")
            .dropna()
            .astype(int)
            .drop_duplicates()
            .tolist()
        )

        if limite is not None:
            ids = ids[:int(limite)]

        detalhes = {}

        for indice, personagem_id in enumerate(ids, start=1):

            arquivo = pasta / f"personagem_{personagem_id}.json"

            if arquivo.exists():
                try:
                    with open(arquivo, "r", encoding="utf-8") as f:
                        detalhe = json.load(f)

                    if int(detalhe.get("id", -1)) == personagem_id:
                        detalhes[str(personagem_id)] = detalhe
                        continue
                except (ValueError, OSError, AttributeError):
                    pass

            try:
                detalhe = self.buscar_detalhes_personagem(personagem_id)
            except RuntimeError as erro:
                print(f"Interrompido no personagem {personagem_id}: {erro}")
                print("O cache permite continuar em outra execução.")
                break

            if not detalhe or int(detalhe.get("id", -1)) != personagem_id:
                print(f"Detalhes não encontrados para o ID {personagem_id}.")
                continue

            temporario = Path(str(arquivo) + ".tmp")

            with open(temporario, "w", encoding="utf-8") as f:
                json.dump(detalhe, f, ensure_ascii=False, indent=2)

            os.replace(temporario, arquivo)
            detalhes[str(personagem_id)] = detalhe

            if indice % 10 == 0 or indice == len(ids):
                print(f"Detalhes consultados: {indice}/{len(ids)}")

        print(f"Detalhes disponíveis: {len(detalhes)}")
        return detalhes

    # --------------------------------------
    # Volumes (séries) por editora
    # A filtragem por publisher é local.
    # --------------------------------------

    def organizar_volume(self, volume):

        publisher = volume.get("publisher")

        return {
            "volume_id": volume.get("id"),
            "volume_name": volume.get("name"),
            "publisher": self._nome_recurso(publisher),
            "publisher_id": self._id_recurso(publisher),
            "start_year": volume.get("start_year"),
            "count_of_issues": volume.get("count_of_issues"),
            "image_url": self._url_imagem(volume.get("image"))
        }

    def coletar_volumes_editoras(
        self,
        editoras=(31, 10),
        quantidade_por_editora=20,
        max_paginas=20,
        nome_arquivo="volumes_editoras.csv",
        retomar=True
    ):

        editoras = [int(x) for x in editoras]
        quantidade_por_editora = int(quantidade_por_editora)

        if quantidade_por_editora <= 0:
            raise ValueError("quantidade_por_editora deve ser maior que zero.")

        caminho_csv = self.pasta / nome_arquivo
        caminho_checkpoint = Path(str(caminho_csv) + ".json")

        volumes = []
        ids_coletados = set()
        offset = 0

        if retomar and caminho_csv.exists() and caminho_checkpoint.exists():
            try:
                with open(caminho_checkpoint, "r", encoding="utf-8") as f:
                    progresso = json.load(f)

                if sorted(progresso.get("editoras", [])) != sorted(editoras):
                    raise ValueError("Checkpoint pertence a outro conjunto de editoras.")

                df_existente = self._ler_csv_existente(caminho_csv)

                obrigatorias = {"volume_id", "publisher_id"}
                if not obrigatorias.issubset(df_existente.columns):
                    raise ValueError("CSV de volumes sem volume_id/publisher_id.")

                ids_pub = pd.to_numeric(df_existente["publisher_id"], errors="coerce")
                if ids_pub.isna().any() or not set(ids_pub.astype(int)).issubset(set(editoras)):
                    raise ValueError("CSV de volumes contém editoras inválidas.")

                df_existente = df_existente.drop_duplicates(
                    subset="volume_id",
                    keep="first"
                )

                volumes = df_existente.to_dict(orient="records")
                ids_coletados = set(df_existente["volume_id"].astype(str))
                offset = int(progresso.get("offset", -1))

                if offset < 0:
                    raise ValueError("Offset de volumes inválido.")

                print(f"Retomando catálogo de volumes no offset {offset}.")

            except Exception as erro:
                print(f"Não foi possível retomar volumes: {erro}")
                self._fazer_backup(caminho_csv)
                self._fazer_backup(caminho_checkpoint)
                volumes, ids_coletados, offset = [], set(), 0
        else:
            self._fazer_backup(caminho_csv)
            self._fazer_backup(caminho_checkpoint)

        contagens = {
            editora: sum(
                1 for v in volumes
                if int(v.get("publisher_id", -1)) == editora
            )
            for editora in editoras
        }

        paginas_nesta_execucao = 0

        while any(
            contagens[e] < quantidade_por_editora
            for e in editoras
        ):

            if max_paginas is not None and paginas_nesta_execucao >= max_paginas:
                print("Limite de páginas desta execução atingido.")
                break

            parametros = {
                "limit": 100,
                "offset": offset,
                "sort": "count_of_issues:desc",
                "field_list": (
                    "id,name,publisher,start_year,"
                    "count_of_issues,image"
                )
            }

            print(f"Buscando volumes a partir do offset {offset}...")

            try:
                resultados = self._requisitar("volumes/", parametros)
            except RuntimeError as erro:
                print(f"Coleta de volumes interrompida: {erro}")
                break

            if not isinstance(resultados, list) or not resultados:
                print("Fim dos volumes disponíveis.")
                break

            novos = []
            itens_processados = 0

            for indice, volume in enumerate(resultados):
                itens_processados = indice + 1

                if not isinstance(volume, dict):
                    continue

                publisher = volume.get("publisher")
                if not isinstance(publisher, dict):
                    continue

                try:
                    publisher_id = int(publisher.get("id"))
                except (TypeError, ValueError):
                    continue

                if publisher_id not in editoras:
                    continue

                if contagens[publisher_id] >= quantidade_por_editora:
                    continue

                volume_id = volume.get("id")
                if volume_id is None:
                    continue

                volume_id = str(volume_id)

                if volume_id in ids_coletados:
                    continue

                registro = self.organizar_volume(volume)
                novos.append(registro)
                ids_coletados.add(volume_id)
                contagens[publisher_id] += 1

                if all(
                    contagens[e] >= quantidade_por_editora
                    for e in editoras
                ):
                    break

            if novos:
                df_novos = pd.DataFrame(novos)
                arquivo_existe = caminho_csv.exists()

                df_novos.to_csv(
                    caminho_csv,
                    mode="a" if arquivo_existe else "w",
                    header=not arquivo_existe,
                    index=False,
                    sep=";",
                    encoding="utf-8"
                )

                volumes.extend(novos)

            offset += itens_processados
            paginas_nesta_execucao += 1

            self._salvar_checkpoint(
                caminho_checkpoint,
                {
                    "editoras": editoras,
                    "offset": offset,
                    "contagens": contagens
                }
            )

            print(f"Volumes encontrados por editora: {contagens}")

            if all(
                contagens[e] >= quantidade_por_editora
                for e in editoras
            ):
                break

            if len(resultados) < 100:
                print("Última página do catálogo de volumes.")
                break

        return pd.DataFrame(volumes)

    # --------------------------------------
    # Detalhes dos volumes e conceitos
    # O endpoint de volume utiliza o prefixo 4050.
    # --------------------------------------

    def buscar_detalhes_volume(self, volume_id):

        field_list = (
            "id,name,publisher,start_year,"
            "count_of_issues,concepts"
        )

        resultado = self._requisitar(
            f"volume/4050-{int(volume_id)}/",
            {"field_list": field_list}
        )

        if isinstance(resultado, list):
            return resultado[0] if resultado else None

        if isinstance(resultado, dict):
            return resultado

        return None

    def enriquecer_volumes_com_conceitos(
        self,
        volumes,
        pasta_cache="detalhes_volumes"
    ):

        if not isinstance(volumes, pd.DataFrame):
            raise TypeError("Informe um DataFrame de volumes.")

        pasta = self.pasta / pasta_cache
        pasta.mkdir(parents=True, exist_ok=True)

        detalhes = {}

        for indice, volume in enumerate(
            volumes.to_dict(orient="records"),
            start=1
        ):

            volume_id = int(volume["volume_id"])
            arquivo = pasta / f"volume_{volume_id}.json"

            if arquivo.exists():
                try:
                    with open(arquivo, "r", encoding="utf-8") as f:
                        detalhe = json.load(f)

                    if int(detalhe.get("id", -1)) == volume_id:
                        detalhes[str(volume_id)] = detalhe
                        continue
                except (ValueError, OSError, AttributeError):
                    pass

            try:
                detalhe = self.buscar_detalhes_volume(volume_id)
            except RuntimeError as erro:
                print(f"Interrompido no volume {volume_id}: {erro}")
                print("Os detalhes já salvos em cache serão reaproveitados.")
                break

            if not detalhe or int(detalhe.get("id", -1)) != volume_id:
                print(f"Detalhes não encontrados para o volume {volume_id}.")
                continue

            temporario = Path(str(arquivo) + ".tmp")

            with open(temporario, "w", encoding="utf-8") as f:
                json.dump(detalhe, f, ensure_ascii=False, indent=2)

            os.replace(temporario, arquivo)
            detalhes[str(volume_id)] = detalhe

            if indice % 10 == 0 or indice == len(volumes):
                print(f"Volumes detalhados: {indice}/{len(volumes)}")

        print(f"Volumes detalhados disponíveis: {len(detalhes)}")
        return detalhes


## 3. Configuração e teste de conexão

Se estiver usando arquivos já coletados em uma sessão anterior do Colab, mantenha-os na pasta `dados_comicvine` antes de executar a coleta. Os nomes abaixo são os mesmos usados no projeto anterior.

In [ ]:
comic = ComicVine(
    api_key=API_KEY,
    intervalo=5,
    pasta=(PASTA_DADOS)
)

## 4. Coleta de personagens Marvel e DC

Os IDs utilizados são Marvel `31` e DC `10`.

A coleta usa filtragem local pelo `publisher.id`, pois o filtro de relacionamento da listagem geral de personagens não foi confiável nos testes do projeto.

In [ ]:
QUANTIDADE_PERSONAGENS = 500

marvel = comic.coletar_personagens(
    id_editora=31,
    quantidade=QUANTIDADE_PERSONAGENS,
    nome_arquivo="marvel_final.csv",
    retomar=True
)

dc = comic.coletar_personagens(
    id_editora=10,
    quantidade=QUANTIDADE_PERSONAGENS,
    nome_arquivo="dc_final.csv",
    retomar=True
)

print("MARVEL")
display(marvel.head())

print("DC")
display(dc.head())

print("Validação Marvel:")
display(marvel["publisher"].value_counts(dropna=False))

print("Validação DC:")
display(dc["publisher"].value_counts(dropna=False))


Retomando editora 31: 500 personagens já salvos.
Quantidade solicitada já foi coletada.
Retomando editora 10: 500 personagens já salvos.
Quantidade solicitada já foi coletada.
MARVEL


,id,name,real_name,aliases,gender,birth,publisher,publisher_id,origin,deck,image_url,count_of_issue_appearances,first_appeared_in_issue,date_added,date_last_updated
0,1425,Lucas Jackson,NaN,NaN,1,NaN,Marvel,31,Mutant,A mutant born with a kind heart but a hideous ...,https://comicvine.gamespot.com/a/uploads/scale...,3,"I, Monster",2008-06-06 11:26:20,2022-04-20 08:07:39
1,1440,Wolverine,James Howlett,James Howlett\nLogan\nAgent Ten\nBlack Dragon\...,1,NaN,Marvel,31,Mutant,A long-lived mutant with the rage of a beast a...,https://comicvine.gamespot.com/a/uploads/scale...,16966,And the Wind Howls... Wendigo!,2008-06-06 11:25:14,2026-10-26 00:12:29
2,1441,Magneto,Max Eisenhardt,Erik Magnus Lehnsherr\nThe Master of Magnetism...,1,NaN,Marvel,31,Mutant,"Among the most powerful, recognizable, and inf...",https://comicvine.gamespot.com/a/uploads/scale...,6208,X-Men,2008-06-06 11:27:42,2026-09-04 14:26:57
3,1442,Captain America,Steven Grant Rogers,Steve Rogers\nThe Sentinel of Liberty\nSuper-S...,1,"Sep 28, 1922",Marvel,31,Human,"During World War II, Steve Rogers volunteered ...",https://comicvine.gamespot.com/a/uploads/scale...,12310,Meet Captain America,2008-06-06 11:27:42,2026-10-28 14:32:22
4,1443,Spider-Man,Peter Benjamin Parker,Peter Parker\nCaptain Universe\nDusk\nHornet\n...,1,"Oct 14, 1962",Marvel,31,Human,Peter Parker was bitten by a radioactive spide...,https://comicvine.gamespot.com/a/uploads/scale...,18160,Spider-Man!; The Bell Ringer; The Man in the M...,2008-06-06 11:27:11,2026-09-30 17:09:13


DC


,id,name,real_name,aliases,gender,birth,publisher,publisher_id,origin,deck,image_url,count_of_issue_appearances,first_appeared_in_issue,date_added,date_last_updated
0,1253,Lightning Lad,Garth Ranzz,Garth Ranzz\nLightning Boy\nProty\nLive Wire\n...,1,NaN,DC Comics,10,Alien,Garth Ranzz is one of the three original membe...,https://comicvine.gamespot.com/a/uploads/scale...,1096,The Legion of Super-Heroes,2008-06-06 11:27:37,2026-09-25 08:41:52
1,1254,Dream Girl,Nura Nal,Nura Nal\nDreamy\nDreamer\nMiss Terious\nNura ...,2,NaN,DC Comics,10,Alien,"Nura Nal is from the planet Naltor, where all ...",https://comicvine.gamespot.com/a/uploads/scale...,467,The Menace of Dream Girl!,2008-06-06 11:27:38,2025-06-10 00:18:11
2,1255,Brainiac 5,Querl Dox,Querl Dox\nBrainiac 5.1\nBrainy\nBrainiac \nB-...,1,NaN,DC Comics,10,Alien,Brainiac 5 is a long-standing member of the Le...,https://comicvine.gamespot.com/a/uploads/scale...,1160,The War Between Supergirl and The Supermen Eme...,2008-06-06 11:27:37,2026-09-25 08:38:57
3,1256,Invisible Kid,Lyle Norg,Lyle Norg\nBrainiac 6,1,NaN,DC Comics,10,Human,"He is a member of the Legion of Super-Heroes, ...",https://comicvine.gamespot.com/a/uploads/scale...,436,Hercules in the 20th Century!,2008-06-06 11:27:34,2025-03-24 00:32:13
4,1257,Phantom Girl,Tinya Wazzo,Tinya Wazzo\nTinya Wazzo-Nah\nPhantom Woman\nP...,2,NaN,DC Comics,10,Alien,Tinya Wazzo is a native of the extra-dimension...,https://comicvine.gamespot.com/a/uploads/scale...,780,The War Between Supergirl and The Supermen Eme...,2008-06-06 11:27:38,2025-03-04 10:22:15


Validação Marvel:


,count
publisher,
Marvel,500


Validação DC:


,count
publisher,
DC Comics,500


## 5. Preparação da tabela principal de personagens

A tabela analítica mantém informações curtas e úteis para o dashboard. O resumo `deck` é preservado e limpo. A URL da imagem é mantida em `image_url`.

In [ ]:
def preparar_base_personagens(df):

    if not isinstance(df, pd.DataFrame):
        raise TypeError("A entrada precisa ser um DataFrame.")

    df = df.copy()

    # Compatibilidade com coletas antigas que chamavam a coluna `image`
    if "image_url" not in df.columns and "image" in df.columns:
        df = df.rename(columns={"image": "image_url"})

    colunas = [
        "id",
        "name",
        "real_name",
        "aliases",
        "publisher",
        "publisher_id",
        "gender",
        "birth",
        "origin",
        "count_of_issue_appearances",
        "first_appeared_in_issue",
        "deck",
        "image_url",
        "date_added",
        "date_last_updated"
    ]

    # Campos complementares podem não existir em coletas antigas.
    # Criamos a coluna vazia, mas exigimos os identificadores essenciais.
    essenciais = ["id", "name", "publisher", "publisher_id"]
    faltantes = [
        coluna for coluna in essenciais
        if coluna not in df.columns
    ]

    if faltantes:
        raise ValueError(f"Colunas essenciais ausentes: {faltantes}")

    for coluna in colunas:
        if coluna not in df.columns:
            df[coluna] = pd.NA

    base = df[colunas].copy()

    base = base.dropna(subset=["id"])
    base = base.drop_duplicates(subset="id", keep="first")

    for coluna in [
        "id",
        "publisher_id",
        "count_of_issue_appearances"
    ]:
        base[coluna] = pd.to_numeric(
            base[coluna],
            errors="coerce"
        )

    def limpar_texto(valor):

        if pd.isna(valor):
            return ""

        valor = html.unescape(str(valor))
        valor = re.sub(r"<[^>]*>", " ", valor)
        valor = re.sub(r"\s+", " ", valor).strip()
        return valor

    base["deck"] = base["deck"].apply(limpar_texto).str.slice(0, 32000)
    base["aliases"] = base["aliases"].apply(limpar_texto).str.slice(0, 32000)

    return base.sort_values(
        "name",
        na_position="last"
    ).reset_index(drop=True)


base_marvel = preparar_base_personagens(marvel)
base_dc = preparar_base_personagens(dc)

base_completa = pd.concat(
    [base_marvel, base_dc],
    ignore_index=True
).drop_duplicates(subset="id", keep="first")

print("Personagens por editora:")
display(
    base_completa.groupby(
        ["publisher", "publisher_id"],
        dropna=False
    ).size().reset_index(name="quantidade")
)

print("Prévia da tabela principal:")
display(base_completa.head())


Personagens por editora:


,publisher,publisher_id,quantidade
0,DC Comics,10,500
1,Marvel,31,500


Prévia da tabela principal:


,id,name,real_name,aliases,publisher,publisher_id,gender,birth,origin,count_of_issue_appearances,first_appeared_in_issue,deck,image_url,date_added,date_last_updated
0,3322,"3-D Man (Garrett, Jr.)","Delroy Garrett, Jr.",Triathlon Three Dimensional Man Delroy Garrett...,Marvel,31,1,NaN,Human,211,Turbulence,Delroy has three times the physical abilities ...,https://comicvine.gamespot.com/a/uploads/scale...,2008-06-06 11:27:30,2026-02-08 18:51:58
1,2346,Abdul Kazir,NaN,,Marvel,31,1,NaN,Human,8,NaN,,https://comicvine.gamespot.com/a/uploads/scale...,2008-06-06 11:27:03,2011-05-05 16:42:36
2,2274,Abel Stack,NaN,Doctor Abel Stack Dr. Abel Stack,Marvel,31,1,NaN,Human,22,The Capture Of X-51,,https://comicvine.gamespot.com/a/uploads/scale...,2008-06-06 11:26:15,2013-04-18 22:12:48
3,3489,Abomination,Emil Blonsky,Emil Blonsky Agent R-7 Tyrannus The Ravager of...,Marvel,31,1,NaN,Radiation,571,To Be Beaten By Byrrah!; The Abomination!,"Craving for the power of a Hulk, Emil Blonsky ...",https://comicvine.gamespot.com/a/uploads/scale...,2008-06-06 11:27:38,2026-05-09 21:38:28
4,1779,Absorbing Man,Carl Creel,Carl Creel Crusher Rocky Davis Lightningbolt C...,Marvel,31,1,NaN,Human,821,"The Stronger I Am, the Sooner I Die!","An extremely violent boxer turned criminal, Lo...",https://comicvine.gamespot.com/a/uploads/scale...,2008-06-06 11:27:38,2025-06-29 08:21:28


## 6. Enriquecimento de personagens: equipes, poderes e criadores

Cada detalhe é consultado individualmente e armazenado em cache JSON. Para o primeiro teste, use 10 por editora. Depois de validar os resultados, altere o limite para `None` para consultar todos os personagens — isso pode exigir muitas requisições e deve ser feito gradualmente.

Não são coletados aliados, inimigos ou redes de relacionamentos.

In [ ]:
LIMITE_DETALHES_POR_EDITORA = 500  # Use None para todos os personagens

detalhes_marvel = comic.enriquecer_personagens(
    base_marvel,
    limite=LIMITE_DETALHES_POR_EDITORA
)

detalhes_dc = comic.enriquecer_personagens(
    base_dc,
    limite=LIMITE_DETALHES_POR_EDITORA
)

detalhes_personagens = {
    **detalhes_marvel,
    **detalhes_dc
}

print("Personagens com detalhes disponíveis:", len(detalhes_personagens))


Detalhes disponíveis: 500
Detalhes disponíveis: 500
Personagens com detalhes disponíveis: 1000


## 7. Transformar equipes, poderes e criadores em tabelas

As tabelas ficam normalizadas: uma linha para cada vínculo personagem–equipe, personagem–poder ou personagem–criador.

In [ ]:
def recursos_validos(valor):

    if isinstance(valor, dict):
        valor = [valor]

    if not isinstance(valor, list):
        return []

    return [
        item for item in valor
        if isinstance(item, dict)
        and item.get("id") is not None
        and item.get("name")
    ]


def criar_tabelas_personagens(base, detalhes):

    equipes = []
    poderes = []
    criadores = []

    for _, personagem in base.iterrows():

        personagem_id = int(personagem["id"])
        nome = personagem["name"]

        detalhe = detalhes.get(str(personagem_id), {})

        for equipe in recursos_validos(detalhe.get("teams")):
            equipes.append({
                "character_id": personagem_id,
                "character_name": nome,
                "team_id": equipe["id"],
                "team_name": equipe["name"]
            })

        for poder in recursos_validos(detalhe.get("powers")):
            poderes.append({
                "character_id": personagem_id,
                "character_name": nome,
                "power_id": poder["id"],
                "power_name": poder["name"]
            })

        for criador in recursos_validos(detalhe.get("creators")):
            criadores.append({
                "character_id": personagem_id,
                "character_name": nome,
                "creator_id": criador["id"],
                "creator_name": criador["name"]
            })

    tabelas = {
        "character_teams": pd.DataFrame(
            equipes,
            columns=[
                "character_id", "character_name",
                "team_id", "team_name"
            ]
        ),
        "character_powers": pd.DataFrame(
            poderes,
            columns=[
                "character_id", "character_name",
                "power_id", "power_name"
            ]
        ),
        "character_creators": pd.DataFrame(
            criadores,
            columns=[
                "character_id", "character_name",
                "creator_id", "creator_name"
            ]
        )
    }

    for nome, tabela in tabelas.items():
        tabelas[nome] = tabela.drop_duplicates()

    return tabelas


tabelas_personagens = criar_tabelas_personagens(
    base_completa,
    detalhes_personagens
)

for nome, tabela in tabelas_personagens.items():
    print(nome, tabela.shape)
    display(tabela.head())


character_teams (5437, 4)


,character_id,character_name,team_id,team_name
0,3322,"3-D Man (Garrett, Jr.)",40959,Agents of Atlas
1,3322,"3-D Man (Garrett, Jr.)",63086,Atlas Foundation
2,3322,"3-D Man (Garrett, Jr.)",3806,Avengers
3,3322,"3-D Man (Garrett, Jr.)",64097,Orchis
4,3322,"3-D Man (Garrett, Jr.)",58900,Outlaw Avengers


character_powers (9014, 4)


,character_id,character_name,power_id,power_name
0,3322,"3-D Man (Garrett, Jr.)",2,Super Strength
1,3322,"3-D Man (Garrett, Jr.)",3,Super Speed
2,3322,"3-D Man (Garrett, Jr.)",4,Agility
3,3322,"3-D Man (Garrett, Jr.)",5,Stamina
4,3322,"3-D Man (Garrett, Jr.)",15,Healing


character_creators (1417, 4)


,character_id,character_name,creator_id,creator_name
0,3322,"3-D Man (Garrett, Jr.)",2913,George Pérez
1,3322,"3-D Man (Garrett, Jr.)",41741,Kurt Busiek
2,2274,Abel Stack,5614,Jack Kirby
3,3489,Abomination,1764,Gil Kane
4,3489,Abomination,40467,Stan Lee


## 8. Coleta de séries (Volumes) das editoras

Os `concepts` são disponibilizados no detalhe de um volume. Primeiro coletamos um conjunto de séries por editora, filtrando localmente pelo `publisher.id`.

Para começar, a configuração coleta até 20 volumes por editora, procurando em até 20 páginas do catálogo global. Se o limite de páginas for atingido antes da meta, o notebook informa quantos foram encontrados. O checkpoint permite continuar em outra execução.

In [ ]:
VOLUMES_POR_EDITORA = 500
MAX_PAGINAS_CATALOGO = 500

volumes = comic.coletar_volumes_editoras(
    editoras=(31, 10),
    quantidade_por_editora=VOLUMES_POR_EDITORA,
    max_paginas=MAX_PAGINAS_CATALOGO,
    nome_arquivo="volumes_marvel_dc.csv",
    retomar=True
)

print("Volumes por editora:")
if not volumes.empty:
    display(
        volumes.groupby(
            ["publisher", "publisher_id"]
        ).size().reset_index(name="quantidade")
    )

display(volumes.head())


Retomando catálogo de volumes no offset 2035.
Volumes por editora:


,publisher,publisher_id,quantidade
0,DC Comics,10,500
1,Marvel,31,500


,volume_id,volume_name,publisher,publisher_id,start_year,count_of_issues,image_url
0,771,Movie Comics,DC Comics,10,1939,6,https://comicvine.gamespot.com/a/uploads/scale...
1,773,Superman,DC Comics,10,1939,488,https://comicvine.gamespot.com/a/uploads/scale...
2,796,Batman,DC Comics,10,1940,716,https://comicvine.gamespot.com/a/uploads/scale...
3,832,The Human Torch,Marvel,31,1940,38,https://comicvine.gamespot.com/a/uploads/scale...
4,921,Boy Commandos,DC Comics,10,1942,36,https://comicvine.gamespot.com/a/uploads/scale...


## 9. Buscar os conceitos temáticos dos volumes

O recurso `volume` possui o campo `concepts`, que lista conceitos associados à série. Para consultar esse campo, usamos o endpoint individual `/volume/4050-{id}/`.

A tabela temática será uma relação entre volumes e conceitos. Um volume pode ter vários conceitos, e um conceito pode aparecer em vários volumes.

In [ ]:
detalhes_volumes = comic.enriquecer_volumes_com_conceitos(
    volumes
)

print("Volumes com detalhes temáticos:", len(detalhes_volumes))


Volumes detalhados disponíveis: 1000
Volumes com detalhes temáticos: 1000


## 10. Criar tabelas de volumes e conceitos

In [ ]:
def criar_tabelas_tematicas(volumes, detalhes_volumes):

    if volumes.empty:
        tabela_volumes = pd.DataFrame(columns=[
            "volume_id", "volume_name", "publisher",
            "publisher_id", "start_year",
            "count_of_issues", "image_url"
        ])
        tabela_relacao = pd.DataFrame(columns=[
            "volume_id", "volume_name", "publisher",
            "publisher_id", "concept_id", "concept_name"
        ])
        tabela_conceitos = pd.DataFrame(columns=[
            "concept_id", "concept_name"
        ])
        return tabela_volumes, tabela_relacao, tabela_conceitos

    colunas_volume = [
        "volume_id", "volume_name", "publisher",
        "publisher_id", "start_year",
        "count_of_issues", "image_url"
    ]

    tabela_volumes = volumes[colunas_volume].copy()

    for coluna in ["volume_id", "publisher_id", "count_of_issues"]:
        tabela_volumes[coluna] = pd.to_numeric(
            tabela_volumes[coluna],
            errors="coerce"
        )

    relacoes = []

    for _, volume in volumes.iterrows():

        volume_id = int(volume["volume_id"])
        detalhe = detalhes_volumes.get(str(volume_id), {})

        for conceito in recursos_validos(detalhe.get("concepts")):

            relacoes.append({
                "volume_id": volume_id,
                "volume_name": volume["volume_name"],
                "publisher": volume["publisher"],
                "publisher_id": int(volume["publisher_id"]),
                "concept_id": conceito["id"],
                "concept_name": conceito["name"]
            })

    tabela_relacao = pd.DataFrame(
        relacoes,
        columns=[
            "volume_id",
            "volume_name",
            "publisher",
            "publisher_id",
            "concept_id",
            "concept_name"
        ]
    ).drop_duplicates()

    if tabela_relacao.empty:
        tabela_conceitos = pd.DataFrame(
            columns=["concept_id", "concept_name"]
        )
    else:
        tabela_conceitos = (
            tabela_relacao[["concept_id", "concept_name"]]
            .drop_duplicates(subset="concept_id")
            .sort_values("concept_name")
            .reset_index(drop=True)
        )

    return (
        tabela_volumes.drop_duplicates(subset="volume_id"),
        tabela_relacao,
        tabela_conceitos
    )


base_volumes, volume_concepts, base_concepts = criar_tabelas_tematicas(
    volumes,
    detalhes_volumes
)

print("Volumes:")
display(base_volumes.head())

print("Relação volume-conceito:")
display(volume_concepts.head())

print("Conceitos únicos:")
display(base_concepts.head())


Volumes:


,volume_id,volume_name,publisher,publisher_id,start_year,count_of_issues,image_url
0,771,Movie Comics,DC Comics,10,1939,6,https://comicvine.gamespot.com/a/uploads/scale...
1,773,Superman,DC Comics,10,1939,488,https://comicvine.gamespot.com/a/uploads/scale...
2,796,Batman,DC Comics,10,1940,716,https://comicvine.gamespot.com/a/uploads/scale...
3,832,The Human Torch,Marvel,31,1940,38,https://comicvine.gamespot.com/a/uploads/scale...
4,921,Boy Commandos,DC Comics,10,1942,36,https://comicvine.gamespot.com/a/uploads/scale...


Relação volume-conceito:


,volume_id,volume_name,publisher,publisher_id,concept_id,concept_name
0,773,Superman,DC Comics,10,43734,Homage Covers
1,773,Superman,DC Comics,10,56090,Marriage
2,773,Superman,DC Comics,10,41148,Green Lantern
3,773,Superman,DC Comics,10,57596,Comic Books In Media
4,773,Superman,DC Comics,10,57349,Variant Cover


Conceitos únicos:


,concept_id,concept_name
0,56142,'Nuff Said
1,56595,Art Book
2,56004,Assistant Editors' Month
3,56429,Association Football
4,56520,Auto Racing


## 11. Exportação de todas as tabelas para Power BI

Os arquivos finais usam `;` como separador e `utf-8-sig` para facilitar a importação no Excel/Power BI.

A descrição longa dos personagens não é exportada.

In [ ]:
PASTA_EXPORTACAO = Path("exportacoes_powerbi")
PASTA_EXPORTACAO.mkdir(parents=True, exist_ok=True)

tabelas_exportacao = {
    "characters.csv": base_completa,
    "character_teams.csv": tabelas_personagens["character_teams"],
    "character_powers.csv": tabelas_personagens["character_powers"],
    "character_creators.csv": tabelas_personagens["character_creators"],
    "volumes.csv": base_volumes,
    "volume_concepts.csv": volume_concepts,
    "concepts.csv": base_concepts
}

for nome_arquivo, tabela in tabelas_exportacao.items():

    caminho = PASTA_EXPORTACAO / nome_arquivo

    tabela.to_csv(
        caminho,
        index=False,
        sep=";",
        encoding="utf-8-sig"
    )

    print(f"{nome_arquivo}: {len(tabela)} registros")

print("\nExportação concluída.")


characters.csv: 1000 registros
character_teams.csv: 5437 registros
character_powers.csv: 9014 registros
character_creators.csv: 1417 registros
volumes.csv: 1000 registros
volume_concepts.csv: 1157 registros
concepts.csv: 149 registros

Exportação concluída.


## 12. Validação final

In [ ]:
print("PERSONAGENS POR EDITORA")
display(
    base_completa.groupby(
        ["publisher", "publisher_id"]
    ).agg(
        personagens=("id", "nunique"),
        aparicoes=("count_of_issue_appearances", "sum")
    ).reset_index()
)

print("VOLUMES POR EDITORA")
display(
    base_volumes.groupby(
        ["publisher", "publisher_id"]
    ).size().reset_index(name="volumes")
)

print("CONCEITOS POR EDITORA")
if not volume_concepts.empty:
    display(
        volume_concepts.groupby(
            ["publisher", "publisher_id"]
        )["concept_id"].nunique().reset_index(
            name="conceitos_distintos"
        )
    )

print("IDs duplicados na tabela de personagens:",
      base_completa["id"].duplicated().sum())


PERSONAGENS POR EDITORA


,publisher,publisher_id,personagens,aparicoes
0,DC Comics,10,500,248599
1,Marvel,31,500,499551


VOLUMES POR EDITORA


,publisher,publisher_id,volumes
0,DC Comics,10,500
1,Marvel,31,500


CONCEITOS POR EDITORA


,publisher,publisher_id,conceitos_distintos
0,DC Comics,10,62
1,Marvel,31,117


IDs duplicados na tabela de personagens: 0


## 13. Compactar os arquivos para download no Colab

A célula cria um ZIP com os CSVs da pasta `exportacoes_powerbi`.

In [ ]:
arquivo_zip = shutil.make_archive(
    "exportacoes_powerbi",
    "zip",
    root_dir=PASTA_EXPORTACAO
)

print("Arquivo compactado:", arquivo_zip)

try:
    from google.colab import files
    files.download(arquivo_zip)
except ImportError:
    print("Fora do Google Colab. O ZIP está salvo no diretório atual.")


Arquivo compactado: /content/exportacoes_powerbi.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 14. Relacionamentos sugeridos no Power BI

Configure os relacionamentos de um para muitos:

- `characters[id]` → `character_teams[character_id]`
- `characters[id]` → `character_powers[character_id]`
- `characters[id]` → `character_creators[character_id]`
- `volumes[volume_id]` → `volume_concepts[volume_id]`
- `concepts[concept_id]` → `volume_concepts[concept_id]`

**Interpretação:** os conceitos temáticos pertencem às séries (volumes) cadastradas no Comic Vine. Não devem ser tratados como atributos diretos de um personagem.

**Cobertura:** os detalhes de personagens e volumes dependem dos limites definidos nas células 6 e 8. Se usar limites de teste, as tabelas complementares representam apenas a amostra enriquecida, não a totalidade do catálogo.

**Uso responsável:** respeite os termos de uso e as condições de atribuição da Comic Vine. A API informa que seu uso é não comercial.